# Tau²-Style Multi-Turn Eval (Eval Protocol)

This notebook is a **repo-grounded template** for multi-turn evals with MCP tools.

It does **not** assume tutorial file paths exist locally. Instead it:
- discovers candidate dataset/server files in this repo,
- asks you to choose explicit paths,
- then generates a pytest eval harness from those real paths.

In [1]:
from pathlib import Path
import urllib.request

# ---- edit these ----
MODEL = "fireworks_ai/accounts/fireworks/models/glm-5p2"
SIM_USER_LLM = "gpt-4.1"
MAX_TOKENS = 4096
TEMPERATURE = 0.0
MAX_CONCURRENT_ROLLOUTS = 16
PASSED_THRESHOLD = 0.4

training_dir = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "training" and (p / "pyproject.toml").exists()),
    Path("../../").resolve(),
)

# Tau2 airline dataset source (user-provided URL)
AIRLINE_DATASET_URL = "https://raw.githubusercontent.com/eval-protocol/python-sdk/1bd5447a3afbca3b71e0f0d205ed7cff6c3afe5d/eval_protocol/benchmarks/data/airline_dataset.jsonl"
DATASET_PATH = "examples/benchmarks/data/airline_dataset.jsonl"
SERVER_SCRIPT_PATH = "examples/rl/eval_protocol_chat/remote_server/server.py"
DOWNLOAD_DATASET_IF_MISSING = True

resolved_dataset = training_dir / DATASET_PATH
if DOWNLOAD_DATASET_IF_MISSING and not resolved_dataset.exists():
    resolved_dataset.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(AIRLINE_DATASET_URL, resolved_dataset)
    print(f"Downloaded dataset -> {resolved_dataset}")

# Repo-local discovery (for validation / visibility)
dataset_candidates = sorted([str(p.relative_to(training_dir)) for p in training_dir.rglob("*.jsonl")])
server_candidates = sorted([str(p.relative_to(training_dir)) for p in training_dir.rglob("*server*.py")])

print("training_dir:", training_dir)
print("\nJSONL candidates:")
for p in dataset_candidates:
    print(" -", p)
print("\nserver.py candidates:")
for p in server_candidates:
    print(" -", p)

assert DATASET_PATH in dataset_candidates, f"DATASET_PATH not found in repo: {DATASET_PATH}"
assert SERVER_SCRIPT_PATH in server_candidates, f"SERVER_SCRIPT_PATH not found in repo: {SERVER_SCRIPT_PATH}"
print("\nUsing DATASET_PATH:", DATASET_PATH)
print("Using SERVER_SCRIPT_PATH:", SERVER_SCRIPT_PATH)


Downloaded dataset -> /Users/sinan/cookbook/training/examples/benchmarks/data/airline_dataset.jsonl
training_dir: /Users/sinan/cookbook/training

JSONL candidates:
 - examples/benchmarks/data/airline_dataset.jsonl
 - examples/orpo/ifeval/dataset.jsonl
 - examples/rl/deepmath/dataset.jsonl
 - examples/rl/eval_protocol_chat/train.jsonl
 - examples/rl/frozen_lake/seeds.jsonl
 - examples/rl/multi_turn_message_in/test.jsonl
 - examples/rl/multi_turn_message_in/train.jsonl
 - examples/sft/food_reasoning.jsonl
 - examples/sft/text2sql_dataset.jsonl

server.py candidates:
 - examples/rl/eval_protocol_chat/remote_server/server.py
 - tests/glm5_serverless_cases.py
 - tests/smoke_test/test_glm5_serverless_prompt_tokens.py
 - utils/serverless.py

Using DATASET_PATH: examples/benchmarks/data/airline_dataset.jsonl
Using SERVER_SCRIPT_PATH: examples/rl/eval_protocol_chat/remote_server/server.py


In [ ]:
import textwrap

TEST_PATH = training_dir / "tests/pytest/test_multiturn_mcp_from_notebook.py"
TEST_PATH.parent.mkdir(parents=True, exist_ok=True)

resolved_dataset_path = (training_dir / DATASET_PATH).resolve()
resolved_server_path = (training_dir / SERVER_SCRIPT_PATH).resolve()

test_code = textwrap.dedent(f"""
import json
import os
import sys
from pathlib import Path
from typing import Any, Dict, List

from eval_protocol import evaluation_test
from eval_protocol.models import EvaluationRow, InputMetadata

try:
    from eval_protocol.pytest import MCPGymRolloutProcessor as _RolloutProcessor
except Exception:
    from eval_protocol.pytest import default_mcp_gym_rollout_processor as _RolloutProcessor

DATASET_PATH = Path(r\"{str(resolved_dataset_path.as_posix())}\")
SERVER_SCRIPT_PATH = Path(r\"{str(resolved_server_path.as_posix())}\")

assert DATASET_PATH.exists(), f"Dataset path does not exist: {{DATASET_PATH}}"
assert SERVER_SCRIPT_PATH.exists(), f"Server path does not exist: {{SERVER_SCRIPT_PATH}}"

# Ensure eval-protocol server subprocess (which calls plain `python`) uses
# this same interpreter environment.
_shim_dir = Path(__file__).resolve().parent / ".python_shim"
_shim_dir.mkdir(parents=True, exist_ok=True)
_shim_path = _shim_dir / "python"
_shim_path.write_text(
    f"#!/usr/bin/env bash\nexec \"{sys.executable}\" \"$@\"\n",
    encoding="utf-8",
)
os.chmod(_shim_path, 0o755)
os.environ["PATH"] = f"{_shim_dir}:{os.environ.get('PATH', '')}"

def _load_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with path.open(\"r\", encoding=\"utf-8\") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def dataset_to_evaluation_row(data: List[Dict[str, Any]]) -> List[EvaluationRow]:
    rows: List[EvaluationRow] = []
    for i, row in enumerate(data):
        rows.append(EvaluationRow(
            messages=[{{\"role\": \"system\", \"content\": \"You are a helpful agent. Use available tools when needed.\"}}],
            input_metadata=InputMetadata(
                row_id=row.get(\"id\", f\"row-{{i}}\"),
                dataset_info=dict(row),
            ),
        ))
    return rows

@evaluation_test(
    input_dataset=[str(DATASET_PATH)],
    dataset_adapter=dataset_to_evaluation_row,
    completion_params=[{{\"model\": \"{MODEL}\", \"temperature\": {TEMPERATURE}, \"max_tokens\": {MAX_TOKENS}}}],
    rollout_processor=_RolloutProcessor() if callable(_RolloutProcessor) else _RolloutProcessor,
    passed_threshold={PASSED_THRESHOLD},
    num_runs=1,
    mode=\"pointwise\",
    max_concurrent_rollouts={MAX_CONCURRENT_ROLLOUTS},
    server_script_path=str(SERVER_SCRIPT_PATH),
)
def test_multiturn_mcp(row: EvaluationRow) -> EvaluationRow:
    return row
""")

TEST_PATH.write_text(test_code, encoding="utf-8")
print(f"wrote: {TEST_PATH}")
print(f"dataset: {resolved_dataset_path}")
print(f"server:  {resolved_server_path}")


NameError: name '_shim_dir' is not defined

In [8]:
!pip install uvicorn

In [9]:
# Run this to execute the eval test.
!cd "$training_dir" && pytest -q tests/pytest/test_multiturn_mcp_from_notebook.py -s

print("If server startup fails with ModuleNotFoundError (e.g., uvicorn), install dependency above and rerun.")

/opt/homebrew/Caskroom/miniconda/base/envs/cookbook/lib/python3.12/site-packages/eval_protocol/models.py:1156: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class TaskDefinitionModel(BaseModel):
Tip: Run `ep logs` in another terminal to start the local UI before viewing results.
✅ Server started successfully on port 9700
INFO:eval_protocol.mcp.execution.policy:🗄️ Initialized in-memory caching
INFO:eval_protocol.mcp.execution.policy:✅ Initialized LiteLLM policy: fireworks_ai/accounts/fireworks/models/glm-5p2
INFO:eval_protocol.mcp.execution.manager:🚀 Live mode: No recording/playback
INFO:eval_protocol.mcp.execution.manager:🧵 Starting 50 rollouts with max 16 concurrent threads...

  Run 1:   0%|                                     | 0/50 [00:00<?, ?rollout/s]WARNING:eval_protocol.mcp.execution.manager:Fail